In [1]:
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model import recall_at_k, train_model


def make_recall_at_k_eval_metric(fraction: float = 0.01):
    def recall_at_k_eval_metric(
        y_true: np.ndarray | None, y_pred: np.ndarray
    ) -> tuple[str, float, bool]:
        if y_true is None:
            return "recall_top_1_percent", 0.0, True
        return "recall_top_1_percent", recall_at_k(y_true, y_pred, fraction), True

    return recall_at_k_eval_metric


RANDOM_SEED = 1000

In [2]:
df_features = pl.read_parquet("data/features_250k.parquet")
df_labels = pl.read_parquet("data/sample_250k.1L83.p2rank.1.parquet")

df = df_features.join(df_labels, on="id")
df_clean = df.filter(pl.all_horizontal(pl.col("^.*_valid$")))  # only use valid features

df_clean = df_clean.filter(pl.col("affinity_kcal_mol") < 0)

In [3]:
x_descriptors = df_clean["descriptors"].to_numpy()
x_ecfp = df_clean["ecfp"].to_numpy()
x_e3fp = df_clean["e3fp"].to_numpy()
x_pharmacophore = df_clean["pharmacophore_3d"].to_numpy()
x_usrcat = df_clean["usrcat"].to_numpy()

fingerprints = {
    "ecfp": x_ecfp,
    "e3fp": x_e3fp,
    "pharmacophore_3d": x_pharmacophore,
    "usrcat": x_usrcat,
}

y = df_clean["affinity_kcal_mol"].to_numpy()

indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices, train_size=0.8, random_state=RANDOM_SEED
)
train_idx, stop_idx = train_test_split(
    train_idx, train_size=0.8, random_state=RANDOM_SEED
)

In [4]:
def split_fingerprints(fp_dict, idx):
    return {name: arr[idx] for name, arr in fp_dict.items()}


train_fp = split_fingerprints(fingerprints, train_idx)
stop_fp = split_fingerprints(fingerprints, stop_idx)
test_fp = split_fingerprints(fingerprints, test_idx)

train_desc = x_descriptors[train_idx]
stop_desc = x_descriptors[stop_idx]
test_desc = x_descriptors[test_idx]

y_train = y[train_idx]
y_stop = y[stop_idx]
y_test = y[test_idx]

fingerprint_dims = {name: arr.shape[1] for name, arr in fingerprints.items()}

In [5]:
recall_metric = make_recall_at_k_eval_metric(0.01)


def recall_fn(y_true, y_pred):
    return recall_metric(y_true, y_pred)[1]

In [6]:
model, val_recall = train_model(
    train_fp=train_fp,
    train_desc=train_desc,
    train_y=y_train,
    val_fp=stop_fp,
    val_desc=stop_desc,
    val_y=y_stop,
    fingerprint_dims=fingerprint_dims,
    descriptor_dim=x_descriptors.shape[1],
    log_transform={
        "ecfp": False,
        "e3fp": False,
        "pharmacophore_3d": False,
        "usrcat": False,
    },
    ranking_weight=1.5,
    recall_fn=recall_fn,
    patience=10,
)

epoch 1/50 train_loss=11.3936 top1pct_recall=0.3883
epoch 2/50 train_loss=0.9049 top1pct_recall=0.4149
epoch 3/50 train_loss=0.8200 top1pct_recall=0.4388
epoch 4/50 train_loss=0.7666 top1pct_recall=0.4468
epoch 5/50 train_loss=0.7184 top1pct_recall=0.4548
epoch 6/50 train_loss=0.6730 top1pct_recall=0.4574
epoch 7/50 train_loss=0.6384 top1pct_recall=0.4309
epoch 8/50 train_loss=0.6155 top1pct_recall=0.4069
epoch 9/50 train_loss=0.5935 top1pct_recall=0.4229
epoch 10/50 train_loss=0.5734 top1pct_recall=0.4122
epoch 11/50 train_loss=0.5588 top1pct_recall=0.4335
epoch 12/50 train_loss=0.5354 top1pct_recall=0.4069
epoch 13/50 train_loss=0.5089 top1pct_recall=0.4335
epoch 14/50 train_loss=0.4806 top1pct_recall=0.4335
epoch 15/50 train_loss=0.4602 top1pct_recall=0.4495
epoch 16/50 train_loss=0.4443 top1pct_recall=0.4495
early stopping at epoch 16
